
# Employee Salary Analysis and Regression

This notebook analyzes an employee dataset with the following columns:

- `EmployeeID`
- `Name`
- `Department`
- `Experience_Years`
- `Education_Level`
- `Age`
- `Gender`
- `City`
- `Monthly_Salary`

The main goal is to train a **regression model** that predicts `Monthly_Salary` for new employee data.

> Put your dataset CSV in the same folder as this notebook and name it `employees.csv`, or change the path in the loading cell.


## 1. Import libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


## 2. Load the dataset

In [ ]:

# Change this path if your CSV has a different name or location
DATASET_PATH = "employees.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
df.head()


## 3. Basic dataset inspection

In [ ]:

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:

df.describe(include="all").T


## 4. Clean the dataset

In [ ]:

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Ensure the target is numeric
df["Monthly_Salary"] = pd.to_numeric(df["Monthly_Salary"], errors="coerce")

# Remove rows where the target is missing
df = df.dropna(subset=["Monthly_Salary"])

print("Cleaned dataset shape:", df.shape)


## 5. Exploratory data analysis

In [ ]:

plt.figure(figsize=(8, 5))
plt.hist(df["Monthly_Salary"], bins=20, edgecolor="black")
plt.xlabel("Monthly Salary")
plt.ylabel("Frequency")
plt.title("Distribution of Monthly Salary")
plt.show()


In [ ]:

numeric_columns = ["Experience_Years", "Age", "Monthly_Salary"]

existing_numeric = [c for c in numeric_columns if c in df.columns]
correlation = df[existing_numeric].corr()

correlation


In [ ]:

plt.figure(figsize=(7, 5))
plt.scatter(df["Experience_Years"], df["Monthly_Salary"], alpha=0.6)
plt.xlabel("Experience Years")
plt.ylabel("Monthly Salary")
plt.title("Salary vs Experience")
plt.show()


In [ ]:

plt.figure(figsize=(7, 5))
plt.scatter(df["Age"], df["Monthly_Salary"], alpha=0.6)
plt.xlabel("Age")
plt.ylabel("Monthly Salary")
plt.title("Salary vs Age")
plt.show()



## 6. Prepare features and target

`EmployeeID` and `Name` are excluded because they normally identify a person rather than provide useful information for general salary prediction.

The model uses:

- `Department`
- `Experience_Years`
- `Education_Level`
- `Age`
- `Gender`
- `City`

to predict:

- `Monthly_Salary`


In [ ]:

target = "Monthly_Salary"

columns_to_drop = ["EmployeeID", "Name", target]
X = df.drop(columns=columns_to_drop, errors="ignore")
y = df[target]

print("Features used:")
print(X.columns.tolist())


## 7. Identify numerical and categorical columns

In [ ]:

numerical_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


## 8. Create preprocessing pipeline

In [ ]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## 9. Train/test split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


## 10. Build and train the regression model

In [ ]:

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

print("Model training completed.")


## 11. Evaluate the model

In [ ]:

y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")


In [ ]:

results = pd.DataFrame({
    "Actual_Salary": y_test.values,
    "Predicted_Salary": y_pred
})

results["Error"] = results["Actual_Salary"] - results["Predicted_Salary"]
results.head(10)


In [ ]:

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.7)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.xlabel("Actual Monthly Salary")
plt.ylabel("Predicted Monthly Salary")
plt.title("Actual vs Predicted Salary")
plt.show()


## 12. Cross-validation

In [ ]:

cv_r2_scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("R² scores:", cv_r2_scores)
print("Mean R²:", cv_r2_scores.mean())
print("R² standard deviation:", cv_r2_scores.std())


## 13. Inspect feature importance

In [ ]:

# Get transformed feature names
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

# Get Random Forest feature importances
importances = pipeline.named_steps["model"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

feature_importance.head(20)


In [ ]:

top_features = feature_importance.head(15).sort_values("Importance")

plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.title("Top Feature Importances")
plt.tight_layout()
plt.show()



## 14. Predict salary for new employee data

For prediction, provide the same feature columns that were used during training.

You do **not** need to provide `Monthly_Salary`, because that is what the model will predict.


In [ ]:

new_employee = pd.DataFrame([
    {
        "Department": "Engineering",
        "Experience_Years": 5,
        "Education_Level": "Master",
        "Age": 30,
        "Gender": "Male",
        "City": "Milan"
    }
])

new_employee


In [ ]:

predicted_salary = pipeline.predict(new_employee)

print(f"Predicted monthly salary: {predicted_salary[0]:.2f}")


## 15. Predict salaries for multiple new employees

In [ ]:

new_employees = pd.DataFrame([
    {
        "Department": "Engineering",
        "Experience_Years": 5,
        "Education_Level": "Master",
        "Age": 30,
        "Gender": "Male",
        "City": "Milan"
    },
    {
        "Department": "Finance",
        "Experience_Years": 8,
        "Education_Level": "Bachelor",
        "Age": 35,
        "Gender": "Female",
        "City": "Rome"
    }
])

new_employees["Predicted_Monthly_Salary"] = pipeline.predict(new_employees)

new_employees


## 16. Save the trained model

In [ ]:

MODEL_PATH = "salary_regression_model.joblib"

joblib.dump(pipeline, MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")


## 17. Load the saved model and use it later

In [ ]:

loaded_model = joblib.load("salary_regression_model.joblib")

example = pd.DataFrame([
    {
        "Department": "Engineering",
        "Experience_Years": 6,
        "Education_Level": "Master",
        "Age": 32,
        "Gender": "Female",
        "City": "Milan"
    }
])

prediction = loaded_model.predict(example)

print(f"Predicted monthly salary: {prediction[0]:.2f}")



## Notes

- The quality of the predictions depends heavily on the amount and quality of your data.
- `OneHotEncoder(handle_unknown="ignore")` allows the pipeline to handle unseen categories without crashing.
- Random Forest can model nonlinear relationships between employee characteristics and salary.
- For larger datasets, it is worth comparing this model with models such as `GradientBoostingRegressor`, `HistGradientBoostingRegressor`, XGBoost, or LightGBM.
- Avoid using `Name` or `EmployeeID` as predictors because they can cause the model to memorize individuals rather than learn useful salary patterns.
